In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DateType,
    TimestampType
)
from datetime import datetime


# =========================
# Configuration
# =========================

SOURCE_PATH = (
    "abfss://raw@fintechdllasya.dfs.core.windows.net/"
    "accounts/accounts.csv"
)

BRONZE_TABLE = (
    "dbx_fintech_data_platform.bronze.accounts"
)

INGESTION_LOG_TABLE = (
    "dbx_fintech_data_platform.metadata.ingestion_log"
)

PIPELINE_NAME = "account_bronze_ingestion"

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS dbx_fintech_data_platform.metadata
""")

In [0]:
log_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("source_date", DateType(), True),
    StructField("target_table", StringType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("error_message", StringType(), True)
])

if not spark.catalog.tableExists(INGESTION_LOG_TABLE):
    (
        spark.createDataFrame([], log_schema)
        .write
        .format("delta")
        .saveAsTable(INGESTION_LOG_TABLE)
    )

In [0]:
started_at = datetime.now()

existing_count = (
    spark.table(INGESTION_LOG_TABLE)
    .filter(F.col("source_file") == SOURCE_PATH)
    .filter(F.col("status") == "SUCCESS")
    .count()
)

if existing_count > 0:

    print("SKIPPED: Source file already processed.")
    print(SOURCE_PATH)

else:

    print("PROCESSING:", SOURCE_PATH)

    try:

        account_df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(SOURCE_PATH)
        )

        bronze_df = (
            account_df
            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )
            .withColumn(
                "_source_file",
                F.col("_metadata.file_path")
            )
        )

        rows_processed = bronze_df.count()

        (
            bronze_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(BRONZE_TABLE)
        )

        completed_at = datetime.now()

        log_data = [(
            PIPELINE_NAME,
            SOURCE_PATH,
            None,
            BRONZE_TABLE,
            "SUCCESS",
            rows_processed,
            started_at,
            completed_at,
            None
        )]

        (
            spark.createDataFrame(log_data, log_schema)
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(INGESTION_LOG_TABLE)
        )

        print(
            f"SUCCESS: {rows_processed} records processed."
        )

    except Exception as e:

        completed_at = datetime.now()

        error_data = [(
            PIPELINE_NAME,
            SOURCE_PATH,
            None,
            BRONZE_TABLE,
            "FAILED",
            0,
            started_at,
            completed_at,
            str(e)
        )]

        (
            spark.createDataFrame(error_data, log_schema)
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(INGESTION_LOG_TABLE)
        )

        raise

In [0]:
bronze_account_df = spark.table(BRONZE_TABLE)

print(
    "Bronze account records:",
    bronze_account_df.count()
)

print(
    "Unique accounts:",
    bronze_account_df
    .select("account_id")
    .distinct()
    .count()
)

In [0]:
display(
    bronze_account_df
)

In [0]:
display(
    spark.table(INGESTION_LOG_TABLE)
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
)